# Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
from scipy.signal import find_peaks, peak_widths

from VirtualSAS_theory import VirtualSAS_theory

from AFL.double_agent import *
from AFL.double_agent.PyTorchExtrapolator import DirichletGPExtrapolator 

from typing import Optional, Union, List, Tuple, Dict
import textwrap
from matplotlib import gridspec
from scipy.stats import gaussian_kde
from matplotlib.ticker import ScalarFormatter
import matplotlib.transforms as mtransforms 
from matplotlib.colors import Normalize 

from cost import DesignSpaceHierarchyCost, BinaryProbabilityCost, AcquisitionWithCost
from utility import MarginalEntropyAlongDimension, MarginalEntropyOverDimension 
from query_strategy import ArgMaxOverDimension, FullWidthHalfMaximum1D 

### Create Virtual Instrument

In [3]:
from AFL.automation.APIServer.data.DataTrashcan import DataTrashcan

boundary_dataset = xr.load_dataset('250610-extrap_expand_dataset.nc')
boundary_dataset.close()

reference_data_path = './SANS/'
inst_client = VirtualSAS_theory()
inst_client.data = DataTrashcan()
boundary_dataset['labels'] = boundary_dataset.labels.astype(str)
inst_client.boundary_dataset = boundary_dataset
inst_client.trace_boundaries(hull_tracing_ratio=0.25)

# specify reference data
for fname in ['low_q.ABS', 'med_q.ABS', 'high_q.ABS']:
    data = pd.read_csv(str(pathlib.Path(reference_data_path) / fname),sep=r'\s+')#, delim_whitespace=True)
    inst_client.add_configuration(
        q=list(data.q),
        I=list(data.I),
        dI=list(data.dI),
        dq=list(data.dq),
        reset=False
    )

inst_client.add_sasview_model(
    label='2',
    model_name='polymer_excl_volume',
    model_kw={
        'scale': 1.0,
        'background': 1.0,
        'rg': 100.0,
    }
)

inst_client.add_sasview_model(
    label='1',
    model_name='sphere',
    model_kw={
        'scale': 0.5,
        'background': 1.0,
        'sld': 1.0,
        'sld_solvent': 6.0,
        'radius': 10,
    }
)

inst_client.add_sasview_model(
    label='0',
    model_name='power_law',
    model_kw={
        'scale': 1e-7,
        'background': 1.0,
        'power': 4.0,
    }
)
inst_client

### initial Dataset

In [4]:
ds_list = []
for temp in [0,10,20,30,40,50]:
    ds = inst_client.simple_expose({'protein':40,'temperature':temp,'glycerol':5})
    ds = ds.rename({'composition': 'design_space', 'component':'ds_dim'})
    ds['phases'] = int(ds.attrs['labels'])
    ds['step'] = 0
    ds_list.append(ds)

ds_init = xr.concat(ds_list, dim='sample')
ds_init

<xarray.Dataset> Size: 43kB
Dimensions:       (sample: 6, q: 281, ds_dim: 3)
Coordinates:
  * q             (q) float64 2kB 0.003836 0.004348 0.004859 ... 0.6293 0.6334
  * ds_dim        (ds_dim) <U11 132B 'protein' 'temperature' 'glycerol'
Dimensions without coordinates: sample
Data variables:
    I             (sample, q) float64 13kB 1.95 1.937 1.923 ... 1.0 1.0 1.0
    I_noiseless   (sample, q) float64 13kB 1.95 1.937 1.923 ... 1.0 1.0 1.0
    dI            (sample, q) float64 13kB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    design_space  (sample, ds_dim) int64 144B 40 0 5 40 10 5 ... 40 40 5 40 50 5
    phases        (sample) int64 48B 2 1 1 1 0 0
    step          (sample) int64 48B 0 0 0 0 0 0
Attributes:
    components:      ['protein' 'temperature' 'glycerol']
    components_dim:  component
    labels:          2
    labels_dim:      label

In [5]:
def minmax_normalize(arr: np.ndarray,
                    axis: Optional[Union[int, Tuple[int, ...]]] = None,
                    eps: float = 1e-8
                ) -> np.ndarray:
    """
    Min-max normalize a NumPy array along a given axis or globally.

    Parameters:
    -----------
    arr : np.ndarray
        Input array (1D or multi-dimensional).
    axis : Optional[int or Tuple[int, ...]]
        Axis or axes along which to normalize. If None, normalize globally.
    eps : float
        Small epsilon to avoid division by zero.

    Returns:
    --------
    normalized : np.ndarray
        Normalized array with same shape as input, scaled to [0, 1].
    """
    arr = np.asarray(arr)
    min_val = np.min(arr, axis=axis, keepdims=True)
    max_val = np.max(arr, axis=axis, keepdims=True)
    normalized = (arr - min_val) / (max_val - min_val + eps)
    return normalized

### Modeling unknown feasibility constraints

In [7]:
class FeasibilityLabeler(PipelineOp):
    def __init__(
        self,
        input_variable: str = None,
        output_variable: str = None,
        dim: str = "sample",
        name: str = "FeasibilityLabeler",
    ) -> None:
        super().__init__(
            name=name, 
            input_variable=input_variable, 
            output_variable=output_variable,
        )
        self.dim = dim

    def calculate(self, dataset: xr.Dataset) -> Self:
        """Label samples as feasible or infeasible based on cost."""
        labels = dataset[self.input_variable].values
        is_feasible = np.where(labels == 0, 0, 1)  # 1 for feasible, 0 for infeasible
        self.output[self.output_variable] = xr.DataArray(is_feasible, dims=[self.dim])
        self.output[self.output_variable].attrs[
            "description"
        ] = f"Feasibility labels based on {self.input_variable}"

        return self

### Creatte the pipeline for active learning

In [12]:
gp_params_mcmc = {"num_samples":20, "num_warmup":20, "method":"mcmc", "verbose":False}

In [ ]:
with Pipeline(name = "phase_boundaries") as p:
    CartesianGrid(
        output_variable="design_space_grid",
        sample_dim="ds_grid",
        component_dim ="ds_dim",
        grid_spec = {'protein':{'min': 0.0, 'max': 80.0, 'steps': 20},
                     'glycerol':{'min': 0.0, 'max': 12.0, 'steps': 20},
                     'temperature':{'min': 0.0, 'max': 50.0, 'steps': 20}}
    )    
    Standardize(
        input_variable="design_space",
        output_variable="normalized_design_space",
        dim="sample",
        component_dim="ds_dim",
        scale_variable=None,
        min_val={'protein': 0.0, 'glycerol': 0.0, 'temperature': 0.0},
        max_val={'protein': 80.0, 'glycerol': 12.0, 'temperature': 50.0},
        name="Standardize",
    )
    Standardize(
        input_variable="design_space_grid",
        output_variable="normalized_design_space_grid",
        dim="ds_grid",
        component_dim="ds_dim",
        scale_variable=None,
        min_val={'protein': 0.0, 'glycerol': 0.0, 'temperature': 0.0},
        max_val={'protein': 80.0, 'glycerol': 12.0, 'temperature': 50.0},
        name="Standardize",
    )
    DirichletGPExtrapolator(
        feature_input_variable="normalized_design_space",
        predictor_input_variable="phases",
        output_prefix="phase",
        grid_variable="normalized_design_space_grid",
        grid_dim="ds_grid",
        sample_dim="sample",
        component_dim = "ds_dim",
        params=gp_params_mcmc,
        name="DirichletGPExtrapolator-PhaseLabels",
    )
    FeasibilityLabeler(
        input_variable="phases",
        output_variable="feasibility_labels",
        dim="sample",
        name="FeasibilityLabeler",
    )
    DirichletGPExtrapolator(
        feature_input_variable="normalized_design_space",
        predictor_input_variable="feasibility_labels",
        output_prefix="feasibility",
        grid_variable="normalized_design_space_grid",
        grid_dim="ds_grid",
        sample_dim="sample",
        component_dim = "ds_dim",
        params=gp_params_mcmc,
        name="DirichletGPExtrapolator-FeasibilityLabels",
    )
    MarginalEntropyOverDimension(
        input_variable= "phase_y_prob",
        coordinate_dims= ['protein', 'glycerol'],
        component_dim= "ds_dim",
        grid_variable= "design_space_grid",        
        output_variable= "composition_utility",
        name= "UtilityMarginalEntropy",
    )
    DesignSpaceHierarchyCost(
        grid_variable = "design_space_grid",
        grid_dim = "ds_grid",
        component_dim = "ds_dim",
        variables_order = ['protein', 'glycerol'], # from highest to lowest cost
        variables_offsets = [0.0, 0.0],
        output_variable="hierarchy_cost",
        name = "DesignSpaceHierarchyCost",        
    )
    AcquisitionWithCost(
        input_variable = "composition_utility",
        cost_variables = "hierarchy_cost",
        component_dim = None,
        cost_coordinate_dims = None,
        grid_dim = "grid",
        output_variable = "composition_utility_with_cost",
        name = "AcquisitonWithCost",
    )
    ArgMaxOverDimension(
        input_variable="composition_utility_with_cost",
        coordinate_dims= ["protein", "glycerol"],
        output_variable= "next_composition",
        name= "QueryStrategyCompositionSpace",
    )
    MarginalEntropyAlongDimension(
        input_variable= "phase_y_prob",
        conditioning_point= "next_composition",
        complement_coordinate_dims= ['protein', 'glycerol'],
        entropy_coordinate_dim= "temperature",
        grid_variable= "design_space_grid",        
        component_dim= "ds_dim",
        output_variable= "temperature_utility",
        name= "MarginalEntropyAlongDimension",
    )
    BinaryProbabilityCost(
        input_variable="feasibility_y_prob",
        cost_coordinate_dim=""
    )
    FullWidthHalfMaximum1D(
        input_variable= "temperature_utility",
        coordinate_dim= "entropy_dim",
        output_variable= "next_sample",
        name= "FullWidthHalfMaximum1D",
    )
    

p.print()

PipelineOp                               input_variable ---> output_variable
----------                               -----------------------------------
0  ) <CartesianGridGenerator>            CartesianGridGenerator ---> design_space_grid
1  ) <Standardize>                       design_space ---> normalized_design_space
2  ) <Standardize>                       design_space_grid ---> normalized_design_space_grid
3  ) <DirichletGPExtrapolator-PhaseLabels> ['normalized_design_space', 'phases', 'normalized_design_space_grid'] ---> ['phase_y_prob']
4  ) <FeasibilityLabeler>                phases ---> feasibility_labels
5  ) <DirichletGPExtrapolator-FeasibilityLabels> ['normalized_design_space', 'feasibility_labels', 'normalized_design_space_grid'] ---> ['feasibility_y_prob']
6  ) <UtilityMarginalEntropy>            phase_y_prob ---> composition_utility
7  ) <QueryStrategyCompositionSpace>     composition_utility ---> next_composition
8  ) <MarginalEntropyAlongDimension>     phase_y_prob -

In [14]:
ds_result = p.calculate(ds_init)
ds_result

  0%|          | 0/10 [00:00<?, ?it/s]

<xarray.Dataset> Size: 1MB
Dimensions:                       (q: 281, ds_dim: 3, sample: 6, ds_grid: 8000,
                                   phase_n_classes: 3,
                                   feasibility_n_classes: 2, points: 400,
                                   comp_dim: 2, n_counts: 1, entropy_dim: 20,
                                   n_next: 3)
Coordinates:
  * q                             (q) float64 2kB 0.003836 0.004348 ... 0.6334
  * ds_dim                        (ds_dim) object 24B 'protein' ... 'glycerol'
  * points                        (points) int64 3kB 0 1 2 3 ... 396 397 398 399
    protein                       (points) float64 3kB 0.0 0.0 0.0 ... 80.0 80.0
    glycerol                      (points) float64 3kB 0.0 0.6316 ... 11.37 12.0
  * comp_dim                      (comp_dim) <U8 64B 'protein' 'glycerol'
  * entropy_dim                   (entropy_dim) float64 160B 0.0 2.632 ... 50.0
Dimensions without coordinates: sample, ds_grid, phase_n_classes,
                                feasibility_n_classes, n_counts, n_next
Data variables: (12/22)
    I                             (sample, q) float64 13kB 1.95 1.937 ... 1.0
    I_noiseless                   (sample, q) float64 13kB 1.95 1.937 ... 1.0
    dI                            (sample, q) float64 13kB 0.0 0.0 ... 0.0 0.0
    design_space                  (sample, ds_dim) int64 144B 40 0 5 ... 40 50 5
    phases                        (sample) int64 48B 2 1 1 1 0 0
    step                          (sample) int64 48B 0 0 0 0 0 0
    ...                            ...
    feasibility_y_prob            (ds_grid, feasibility_n_classes) float64 128kB ...
    feasibility_entropy_gradient  (ds_grid, ds_dim) float32 96kB -0.206 ... 0...
    composition_utility           (points) float64 3kB 1.064 1.06 ... 1.074
    next_composition              (n_counts, comp_dim) float64 16B 0.0 12.0
    temperature_utility           (entropy_dim) float64 160B 1.073 ... 1.07
    next_sample                   (n_next) float64 24B 41.95 44.74 46.32
Attributes:
    components:      ['protein' 'temperature' 'glycerol']
    components_dim:  component
    labels:          2
    labels_dim:      label